In [5]:
# ===== Real-time Gesture Spotting (Jupyter Notebook, CPU, MediaPipe Hands) =====
# - Fix: Keras 3 loader fails to interpret Orthogonal initializer in .keras
#        => try safe_mode=False, then custom_objects mapping fallback
# - English labels; label_map.json optional.
# - Webcam stream mirrored; overlay text is normal.

import os
import json
import time
import pickle
from collections import deque

import cv2
import numpy as np

# ---- Use MediaPipe solutions submodules only (avoid TF import side-effects) ----
from mediapipe.python.solutions import hands as mp_hands
from mediapipe.python.solutions import drawing_utils as mp_drawing

# =========================
# Paths / Constants
# =========================
ART_DIR   = "./artifacts"
ENC_KERAS = os.path.join(ART_DIR, "encoder_model.keras")
HMM_DIR   = os.path.join(ART_DIR, "hmms")
LABEL_JSON= os.path.join(ART_DIR, "label_map.json")
META_JSON = os.path.join(ART_DIR, "meta.json")  # 선택: latent_dim, max_seq_len, num_features 등

# Spotting hyper-parameters (tune for CPU)
STEP          = 2        # frames hop size
THRESH_DIFF   = 0.0      # gesture_ll - ergodic_ll >= THRESH_DIFF -> positive
MIN_MERGE_GAP = 5        # merge same-label intervals if gap <= this
MIN_INTERVAL  = 5        # drop intervals shorter than this

# Drawing
FONT = cv2.FONT_HERSHEY_SIMPLEX
COLOR_TEXT = (255, 255, 255)
COLOR_BOX  = (40, 40, 40)

# =========================
# Utils
# =========================
def load_label_map():
    """Load English label mapping from label_map.json if exists; else default."""
    default = {
        "1": "hidden", "2": "start", "3": "previous", "4": "next", "5": "stop",
        "6": "first",  "7": "white", "8": "last",     "9": "black","10":"bye"
    }
    if os.path.isfile(LABEL_JSON):
        try:
            with open(LABEL_JSON, "r", encoding="utf-8") as f:
                m = json.load(f)
            return {str(k): str(v) for k, v in m.items()}
        except Exception as e:
            print(f"[WARN] Failed to read {LABEL_JSON}: {e}. Use default English map.")
    return default

def load_meta(path=META_JSON):
    """Optional meta loader; returns dict or {}."""
    if os.path.isfile(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            print(f"[WARN] Failed to read meta.json: {e}")
    return {}

def safe_pad_or_trim(x, T, F):
    """
    x: (t, f) -> (T, F): pad zeros or center-crop time; right-pad feature dimension if needed.
    """
    t, f = x.shape
    # feature align
    if f < F:
        padF = np.zeros((t, F - f), dtype=np.float32)
        x = np.concatenate([x, padF], axis=1)
    elif f > F:
        x = x[:, :F]

    # time align
    if t < T:
        padT = np.zeros((T - t, F), dtype=np.float32)
        x = np.concatenate([x, padT], axis=0)
    elif t > T:
        # center trim
        start = (t - T) // 2
        x = x[start:start+T, :]
    return x.astype(np.float32)

def merge_intervals(intervals, min_gap=0):
    if not intervals:
        return []
    intervals.sort(key=lambda z: z[0])
    out = []
    cs, ce, cl = intervals[0]
    for s,e,l in intervals[1:]:
        if l == cl and s <= ce + min_gap:
            ce = max(ce, e)
        else:
            out.append((cs, ce, cl))
            cs, ce, cl = s, e, l
    out.append((cs, ce, cl))
    return out

def filter_short(intervals, min_len=5):
    return [(s,e,l) for (s,e,l) in intervals if (e-s) >= min_len]

# =========================
# Encoder (tf.keras loader with robust fallbacks)
# =========================
class KerasEncoder:
    """
    Loads encoder_model.keras via tf.keras with robust fallbacks for Keras 3.
    Expects input (T, F) -> outputs (T, D).
    """
    def __init__(self, enc_keras_path: str):
        if not os.path.isfile(enc_keras_path):
            raise FileNotFoundError(f"Encoder not found: {enc_keras_path}")

        # 1) primary attempt: tf.keras with safe_mode=False
        try:
            import tensorflow as tf  # TF 2.16.x recommended
            self.model = tf.keras.models.load_model(
                enc_keras_path, compile=False, safe_mode=False
            )
            ishape = self.model.input_shape  # (None, T, F)
            assert isinstance(ishape, (tuple, list)) and len(ishape) == 3
            self.T = int(ishape[1]); self.F = int(ishape[2])
            print(f"[Encoder] tf.keras loaded (safe_mode=False). Expect (T,F)=({self.T},{self.F})")
            return
        except Exception as e1:
            print(f"[WARN] tf.keras load_model failed: {e1}")

        # 2) fallback: standalone keras loader with custom_objects
        try:
            import tensorflow as tf
            import keras
            custom_objects = {
                # map possible initializer identifiers to real classes
                "Orthogonal": tf.keras.initializers.Orthogonal,
                "orthogonal": tf.keras.initializers.Orthogonal,
            }
            self.model = keras.models.load_model(
                enc_keras_path, compile=False, safe_mode=False,
                custom_objects=custom_objects
            )
            ishape = self.model.input_shape
            assert isinstance(ishape, (tuple, list)) and len(ishape) == 3
            self.T = int(ishape[1]); self.F = int(ishape[2])
            print(f"[Encoder] keras loaded with custom_objects. Expect (T,F)=({self.T},{self.F})")
            return
        except Exception as e2:
            print(f"[ERROR] keras load_model fallback failed: {e2}")

        # 3) last-resort: give a precise instruction to re-save the encoder
        raise ValueError(
            "Failed to load encoder_model.keras.\n"
            "→ Fix: Re-save the encoder with Keras3-friendly format:\n"
            "   model.save('encoder_model.keras', include_optimizer=False)\n"
            "또는 TFLite로 내보내서 사용하세요:\n"
            "   tf.lite.TFLiteConverter.from_keras_model(encoder_model) ..."
        )

    def encode(self, window_tf: np.ndarray) -> np.ndarray:
        """
        window_tf: (t, f) arbitrary -> internally padded/trimmed to (T,F)
        returns: (T, D)
        """
        wf = safe_pad_or_trim(window_tf, self.T, self.F)
        y = self.model.predict(wf[np.newaxis, ...], verbose=0)
        return y[0]

# =========================
# Load HMMs
# =========================
def load_hmms(hmm_dir):
    if not os.path.isdir(hmm_dir):
        raise FileNotFoundError(f"HMM dir not found: {hmm_dir}")
    # ergodic (threshold) model
    ergodic_pkl = os.path.join(hmm_dir, "ergodic.pkl")
    if not os.path.isfile(ergodic_pkl):
        raise FileNotFoundError(f"Missing ergodic model: {ergodic_pkl}")
    with open(ergodic_pkl, "rb") as f:
        ergodic = pickle.load(f)

    # gesture-specific HMMs
    gesture_hmms = {}
    for k in range(1, 11):
        p = os.path.join(hmm_dir, f"hmm_{k}.pkl")
        if not os.path.isfile(p):
            raise FileNotFoundError(f"Missing gesture HMM: {p}")
        with open(p, "rb") as f:
            gesture_hmms[str(k)] = pickle.load(f)

    print(f"[HMM] Loaded ergodic + {len(gesture_hmms)} gesture HMMs.")
    return ergodic, gesture_hmms

# =========================
# Landmark processing
# =========================
def landmarks_to_126(results, width, height):
    """
    Build (126,) = 2 hands * 21 lm * (x,y,z)
    If <2 hands, zero-fill the missing hand.
    """
    feat = []
    hands = results.multi_hand_landmarks or []
    # collect up to 2 hands
    lm_all = []
    for hand in hands[:2]:
        lm = []
        for p in hand.landmark:
            lm.extend([p.x, p.y, p.z])
        lm_all.append(lm)
    # hand1
    if len(lm_all) >= 1:
        feat.extend(lm_all[0])
    else:
        feat.extend([0.0]*63)
    # hand2
    if len(lm_all) >= 2:
        feat.extend(lm_all[1])
    else:
        feat.extend([0.0]*63)

    return np.array(feat, dtype=np.float32)

# =========================
# Sliding-window scoring
# =========================
def score_window(latent_seq, ergodic, gesture_hmms):
    """
    latent_seq: (T, D)
    returns:
      best_label(str), best_diff(float)
    """
    L = latent_seq.shape[0]
    lengths = [L]
    f_ll = ergodic.score(latent_seq, lengths)
    best_label, best_diff = None, -1e9
    for k, hmm in gesture_hmms.items():
        g_ll = hmm.score(latent_seq, lengths)
        diff = g_ll - f_ll
        if diff > best_diff:
            best_diff = diff
            best_label = k
    return best_label, best_diff

# =========================
# Main realtime loop (webcam)
# =========================
def run_realtime():
    # -- Models
    encoder = KerasEncoder(ENC_KERAS)
    T_enc, F_enc = encoder.T, encoder.F
    ergodic, gesture_hmms = load_hmms(HMM_DIR)
    id2name = load_label_map()

    # -- Video
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    if not cap.isOpened():
        raise RuntimeError("Cannot open webcam.")

    # -- MediaPipe Hands
    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    buf = deque(maxlen=T_enc)  # maintain exactly T frames for the encoder
    frames = 0
    last_text = ""
    last_put_ts = 0

    print("[INFO] Press 'q' to exit.")
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # Mirror view for user comfort
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = hands.process(rgb)

        # draw landmarks
        if res.multi_hand_landmarks:
            for hand_lms in res.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

        # build 126-dim feature
        feat126 = landmarks_to_126(res, frame.shape[1], frame.shape[0])  # (126,)
        buf.append(feat126)

        # run every STEP frames once we have T_enc frames
        if len(buf) == T_enc and (frames % STEP == 0):
            window = np.vstack(buf)  # (T_enc, 126)
            latent = encoder.encode(window)  # (T_enc, D)
            best_k, best_diff = score_window(latent, ergodic, gesture_hmms)

            if best_diff >= THRESH_DIFF:
                last_text = id2name.get(best_k, best_k)
                last_put_ts = time.time()

        # draw English text (normal orientation)
        if last_text:
            # keep for 0.8s on screen
            if time.time() - last_put_ts < 0.8:
                (tw, th), bs = cv2.getTextSize(last_text, FONT, 1.2, 2)
                cv2.rectangle(frame, (10, 20-th-10), (10+tw+20, 20+10), COLOR_BOX, -1)
                cv2.putText(frame, last_text, (20, 20), FONT, 1.2, COLOR_TEXT, 2, cv2.LINE_AA)
            else:
                last_text = ""

        cv2.imshow("Real-time Gesture (English)", frame)
        frames += 1

        k = cv2.waitKey(1) & 0xFF
        if k == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    hands.close()
    print("[INFO] Stopped.")

# ============== Execute ==============
run_realtime()


[WARN] tf.keras load_model failed: Could not interpret initializer identifier: {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'seed': None, 'gain': 1.0}, 'registered_name': None, 'shared_object_id': 1420153188848}
[Encoder] keras loaded with custom_objects. Expect (T,F)=(82,126)
[HMM] Loaded ergodic + 10 gesture HMMs.
[INFO] Press 'q' to exit.


C:\Users\james\anaconda3\envs\mediapipe-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
C:\Users\james\anaconda3\envs\mediapipe-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
C:\Users\james\anaconda3\envs\mediapipe-env\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. P

[INFO] Stopped.
